# Step 1 of RAG: Document Loading and Chunking Strategies**RAG** (Retrieval-Augmented Generation) has 4 core steps:1. **Document Loading & Chunking** ← *this notebook*2. Generating Embeddings3. Storing in a Vector Database4. Retrieval + passing context to an LLMBefore we can embed or search anything, we need **text**, split into small, meaningful piecescalled **chunks**. This notebook covers *why* chunking matters and *how* to do it several ways.### What you will learn1. Why we can't just feed a whole document to an embedding model2. Loading a `.txt` document3. Loading a `.pdf` document4. Chunking Strategy 1 — Fixed-size character chunking (from scratch)5. Chunking Strategy 2 — `CharacterTextSplitter` (LangChain)6. Chunking Strategy 3 — `RecursiveCharacterTextSplitter` (LangChain, recommended default)7. Chunking Strategy 4 — Sentence-based chunking8. Chunking Strategy 5 — Token-based chunking (important for LLM token limits)9. Comparing all strategies side by side

## Why Chunking Matters- Embedding models have a **maximum input length** (e.g. ~256-512 tokens for many free models).- A whole book or long article won't fit, and even if it did, one giant embedding would blur together  too many different ideas — making search results useless.- **Chunking** splits a document into smaller pieces so each chunk represents one focused idea,  which produces much more accurate embeddings and search results later on.

### Setup — Install Required Libraries

In [ ]:
!pip install -q langchain langchain-text-splitters tiktoken pypdf fpdf2

## 1. Create a Sample Document (so this notebook runs with zero external downloads)In real projects your document would come from a file, website, or database. Here we generate onelocally so the notebook is 100% reliable and network-independent.

In [ ]:
import osos.makedirs("docs", exist_ok=True)article_text = """Artificial Intelligence: An OverviewArtificial Intelligence, or AI, is the field of computer science focused on building systemsthat can perform tasks which normally require human intelligence. These tasks includeunderstanding language, recognizing images, making decisions, and learning from experience.Machine Learning is a subset of AI where systems learn patterns from data instead of followinghardcoded rules. Instead of programming every decision by hand, we feed the system many examples,and it learns the underlying pattern on its own. This approach powers everything from spam filtersto recommendation systems.Deep Learning takes Machine Learning further by using neural networks with many layers. Theselayered networks can automatically discover complex patterns in large amounts of data, such asrecognizing faces in photos or understanding the meaning of a sentence.Natural Language Processing, or NLP, is the branch of AI that focuses specifically on humanlanguage. NLP powers chatbots, translation tools, and search engines. A key building block ofmodern NLP is the embedding, which converts words or sentences into numeric vectors that capturemeaning.Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with alanguage model. Instead of relying only on what a language model memorized during training, RAGfirst retrieves relevant, up-to-date information from a document collection, then passes thatinformation to the model so it can generate a more accurate, grounded answer.Vector Databases store embeddings and allow fast similarity search across millions of documents.They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendationengines used by companies around the world today."""with open("docs/sample_article.txt", "w") as f:    f.write(article_text)print("Document created: docs/sample_article.txt")print("Total characters:", len(article_text))print("Total words:", len(article_text.split()))

## 2. Loading a `.txt` DocumentThe simplest form of document loading — read the raw text from disk.

In [ ]:
with open("docs/sample_article.txt", "r") as f:    loaded_text = f.read()print("Loaded document preview:\n")print(loaded_text[:300], "...")

## 3. Loading a `.pdf` DocumentMost real-world knowledge bases include PDFs. We'll first generate a simple sample PDF (so thisnotebook needs no external file), then load and extract its text using `pypdf`.

In [ ]:
from fpdf import FPDFpdf = FPDF()pdf.add_page()pdf.set_font("Helvetica", size=12)for line in article_text.split("\n"):    pdf.multi_cell(0, 8, line)pdf.output("docs/sample_article.pdf")print("PDF created: docs/sample_article.pdf")

In [ ]:
from pypdf import PdfReaderreader = PdfReader("docs/sample_article.pdf")pdf_text = ""for page in reader.pages:    pdf_text += page.extract_text() + "\n"print("Pages found:", len(reader.pages))print("\nExtracted PDF text preview:\n")print(pdf_text[:300], "...")

**Discussion Point:** Loading is format-specific (`.txt`, `.pdf`, `.docx`, HTML, etc.) but the goal isalways the same — end up with **plain text** that we can chunk. From here on we'll work with the textwe loaded from the `.txt` file.

## 4. Chunking Strategy 1 — Fixed-Size Character Chunking (From Scratch)The simplest strategy: cut the text every N characters, with some overlap between chunks so wedon't lose context at the boundaries.

In [ ]:
def fixed_size_chunking(text, chunk_size=300, overlap=50):    chunks = []    start = 0    while start < len(text):        end = start + chunk_size        chunks.append(text[start:end])        start += chunk_size - overlap    return chunksfixed_chunks = fixed_size_chunking(loaded_text, chunk_size=300, overlap=50)print("Number of chunks:", len(fixed_chunks))print("\n--- Chunk 1 ---\n", fixed_chunks[0])print("\n--- Chunk 2 ---\n", fixed_chunks[1])

**Discussion Point:** Notice fixed-size chunking can cut a sentence or word right in half — it doesn'tknow anything about paragraph or sentence boundaries. It's simple and fast, but not always ideal.

## 5. Chunking Strategy 2 — `CharacterTextSplitter` (LangChain)LangChain provides ready-made splitters. `CharacterTextSplitter` is the library version of what wejust built by hand — it splits on a chosen separator (like a blank line) and respects `chunk_size`.

In [ ]:
from langchain_text_splitters import CharacterTextSplitterchar_splitter = CharacterTextSplitter(    separator="\n\n",    chunk_size=300,    chunk_overlap=50)char_chunks = char_splitter.split_text(loaded_text)print("Number of chunks:", len(char_chunks))for i, chunk in enumerate(char_chunks):    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")    print(chunk)

## 6. Chunking Strategy 3 — `RecursiveCharacterTextSplitter` (Recommended Default)This is the most commonly used splitter in real RAG systems. It tries to split on paragraph breaksfirst, then sentences, then words — only falling back to a hard character cut as a last resort. Thiskeeps chunks much more semantically coherent.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitterrecursive_splitter = RecursiveCharacterTextSplitter(    chunk_size=300,    chunk_overlap=50,    separators=["\n\n", "\n", ". ", " ", ""])recursive_chunks = recursive_splitter.split_text(loaded_text)print("Number of chunks:", len(recursive_chunks))for i, chunk in enumerate(recursive_chunks):    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")    print(chunk)

**Discussion Point:** Compare this output to Strategy 1. The recursive splitter tends to end chunks atnatural breaks (end of a paragraph or sentence) instead of mid-word — this is why it's the most widelyused default in production RAG pipelines.

## 7. Chunking Strategy 4 — Sentence-Based ChunkingSometimes we want every chunk to contain whole sentences only — useful when precision matters morethan fixed size (e.g. legal or medical documents).

In [ ]:
import redef sentence_chunking(text, sentences_per_chunk=2):    # Simple sentence splitter using punctuation (no extra downloads needed)    sentences = re.split(r'(?<=[.!?])\s+', text.strip())    sentences = [s for s in sentences if s]    chunks = []    for i in range(0, len(sentences), sentences_per_chunk):        chunk = " ".join(sentences[i:i + sentences_per_chunk])        chunks.append(chunk)    return chunkssentence_chunks = sentence_chunking(loaded_text, sentences_per_chunk=2)print("Number of chunks:", len(sentence_chunks))for i, chunk in enumerate(sentence_chunks[:4]):    print(f"\n--- Chunk {i+1} ---")    print(chunk)

## 8. Chunking Strategy 5 — Token-Based ChunkingEmbedding models and LLMs actually have limits measured in **tokens**, not characters (roughly1 token ≈ 4 characters in English). Token-based chunking guarantees we never exceed a model's realinput limit.

In [ ]:
import tiktokenencoding = tiktoken.get_encoding("cl100k_base")def token_chunking(text, max_tokens=60, overlap_tokens=10):    tokens = encoding.encode(text)    chunks = []    start = 0    while start < len(tokens):        end = start + max_tokens        chunk_tokens = tokens[start:end]        chunks.append(encoding.decode(chunk_tokens))        start += max_tokens - overlap_tokens    return chunkstoken_chunks = token_chunking(loaded_text, max_tokens=60, overlap_tokens=10)print("Number of chunks:", len(token_chunks))for i, chunk in enumerate(token_chunks[:3]):    token_count = len(encoding.encode(chunk))    print(f"\n--- Chunk {i+1} ({token_count} tokens) ---")    print(chunk)

## 9. Comparing All Strategies Side by Side

In [ ]:
import pandas as pdcomparison = pd.DataFrame({    "Strategy": [        "Fixed-size (scratch)",        "CharacterTextSplitter",        "RecursiveCharacterTextSplitter",        "Sentence-based",        "Token-based"    ],    "Number of Chunks": [        len(fixed_chunks),        len(char_chunks),        len(recursive_chunks),        len(sentence_chunks),        len(token_chunks)    ],    "Avg Chunk Length (chars)": [        round(sum(len(c) for c in fixed_chunks) / len(fixed_chunks), 1),        round(sum(len(c) for c in char_chunks) / len(char_chunks), 1),        round(sum(len(c) for c in recursive_chunks) / len(recursive_chunks), 1),        round(sum(len(c) for c in sentence_chunks) / len(sentence_chunks), 1),        round(sum(len(c) for c in token_chunks) / len(token_chunks), 1)    ]})comparison

In [ ]:
import matplotlib.pyplot as pltplt.figure(figsize=(8, 5))plt.bar(comparison["Strategy"], comparison["Number of Chunks"])plt.xticks(rotation=30, ha="right")plt.ylabel("Number of Chunks")plt.title("Number of Chunks Produced by Each Strategy")plt.tight_layout()plt.show()

## Key Takeaways (Recap for Students)| Strategy | Best For | Watch Out For ||---|---|---|| Fixed-size (scratch) | Learning the core idea | Can cut sentences/words in half || `CharacterTextSplitter` | Simple, separator-based splitting | Still fairly rigid || `RecursiveCharacterTextSplitter` | Most general-purpose RAG use cases | Slightly more setup, but worth it || Sentence-based | Precision-sensitive domains (legal, medical) | Chunk sizes vary a lot || Token-based | Respecting an embedding/LLM's exact token limit | Requires a tokenizer library |**`chunk_overlap`** (used in most strategies above) keeps a bit of the previous chunk's text at thestart of the next chunk, so an idea that spans a chunk boundary isn't completely lost.This notebook produced chunks — the **next step in RAG** is turning each chunk into an embeddingand storing it in a vector database (see the earlier notebooks in this series).### Practice Exercises1. Change `chunk_size` from 300 to 150 and re-run `RecursiveCharacterTextSplitter` — how does the   number of chunks change?2. Try `sentences_per_chunk=1` and `sentences_per_chunk=4` in the sentence-based strategy and compare.3. Load your own `.txt` or `.pdf` file (upload it to Colab's file panel) and run all 5 strategies on it.